In [1]:
!pip install segmentation_models_pytorch

In [2]:
import os, cv2, time,random, torch
import numpy as np
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.checkpoint import checkpoint
from torch.amp import GradScaler, autocast
import torch.nn.functional as F
import segmentation_models_pytorch as smp
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

# configure

In [3]:
class Config:
    def __init__(self):
        # Paths
        self.data_dir = "/kaggle/input/cxr-all-synthetic-cleaned-traditonal-brightness"
        self.external_data_dir = "/kaggle/input/test-tekno"
        self.model_save_path = "efficient-b1_unet_on_all_synthetic_cleaned_traditional_brightness_data.pth"
        
        # Training Hyperparameters
        self.batch_size = 8
        self.lr = 1e-4
        self.epochs = 100
        self.patience = 10
        self.min_delta = 0.01
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # GradNorm Parameters
        self.gradnorm_alpha = 1.0
        self.gradnorm_lr = 0.1
        self.target_rates = [1.0, 1.0, 1.0]  # Equal target rates for [seg, ctr, cardio]
        
        # Data Split (Train/Val/Test)
        self.train_ratio = 0.75
        self.val_ratio = 0.15
        self.test_ratio = 0.10

        # Reproducibility Seeds
        self.seed = 42
        self.set_seeds()

    def set_seeds(self):
        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(self.seed)
            torch.cuda.manual_seed_all(self.seed)
            # These two lines below are important for deterministic CUDA operations
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

# tools

In [4]:
class Dataset(Dataset):
    def __init__(self, image_dir, mask_dir=None, csv_path=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.csv_path = csv_path
        self.image_files = sorted(os.listdir(image_dir))
        
        # Load CTR from CSV if provided
        self.ctr_df = pd.read_csv(csv_path) if csv_path else None

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        base_name = os.path.splitext(img_name)[0]
        
        # Load Image (Grayscale -> 3-channel)
        img_path = os.path.join(self.image_dir, img_name)
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        image = np.repeat(image[..., np.newaxis], 3, axis=-1)  # (H, W, 3)
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        
        # Initialize targets
        mask = None
        ctr_value = 0.0
        cardio_label = 0.0

        # Initialize empty mask tensor if no masks available
        h, w = image.shape[1], image.shape[2]
        empty_mask = torch.zeros((h, w), dtype=torch.long)
        
        # Case 1: Masks available (main dataset)
        if self.mask_dir:
            mask_path = os.path.join(self.mask_dir, f"{base_name}.png")
            if os.path.exists(mask_path):
                mask_img = cv2.imread(mask_path, cv2.IMREAD_COLOR)
                mask = self._convert_mask(mask_img)  # (H, W) with classes 0,1,2
                mask = torch.from_numpy(mask).long()
                ctr_value = self._calculate_ctr_from_mask(mask.numpy())
                cardio_label = 1.0 if ctr_value > 0.5 else 0.0
                return image, mask, torch.tensor(ctr_value), torch.tensor(cardio_label)
        
        # Case 2: CTR from CSV (external dataset)
        if self.ctr_df is not None:
            row = self.ctr_df[self.ctr_df["name"] == base_name]
            if not row.empty:
                ctr_value = row["CTR"].values[0]
                cardio_label = 1.0 if ctr_value > 0.5 else 0.0
                return image, empty_mask, torch.tensor(ctr_value), torch.tensor(cardio_label)
        
        # Default return (no masks, no CSV data)
        return image, empty_mask, torch.tensor(0.0), torch.tensor(0.0)

    def _convert_mask(self, mask_img):
        """Convert RGB mask to class indices: 
           Background=0, Lung=1, Heart=2"""
        h, w = mask_img.shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)
        # Left lung (85,85,85) / Right lung (170,170,170) -> Class 1
        mask[np.all(mask_img == [85, 85, 85], axis=-1)] = 1
        mask[np.all(mask_img == [170, 170, 170], axis=-1)] = 1
        # Heart (255,255,255) -> Class 2
        mask[np.all(mask_img == [255, 255, 255], axis=-1)] = 2
        return mask

    def _calculate_ctr_from_mask(self, mask):
        """Compute CTR from ground truth mask (numpy array)"""
        # Heart (Class 2)
        heart_mask = (mask == 2)
        heart_coords = np.where(heart_mask)
        cardiac_width = np.max(heart_coords[1]) - np.min(heart_coords[1]) + 1 if len(heart_coords[1]) > 0 else 0
        
        # Lung (Class 1)
        lung_mask = (mask == 1)
        lung_coords = np.where(lung_mask)
        thoracic_width = 0
        if len(lung_coords[0]) > 0:
            all_y = np.unique(lung_coords[0])
            max_distance = 0
            for y in all_y:
                x_vals = lung_coords[1][lung_coords[0] == y]
                if len(x_vals) > 0:
                    width = np.max(x_vals) - np.min(x_vals) + 1
                    max_distance = max(max_distance, width)
            thoracic_width = max_distance
        
        return cardiac_width / thoracic_width if thoracic_width > 0 else 0.0

In [5]:
class MultiTaskUNet(nn.Module):
    def __init__(self, encoder_name="efficientnet-b1", num_classes=3):
        super(MultiTaskUNet, self).__init__()
        
        # Shared encoder (EfficientNet-B1)
        self.encoder = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights="imagenet",
            in_channels=3,
            classes=num_classes,
        )
        
        # Get encoder output channels for custom heads
        encoder_channels = self.encoder.encoder.out_channels
        decoder_channels = self.encoder.decoder.blocks[-1].conv2[0].out_channels
        
        # Segmentation head (already included in base U-Net)
        self.segmentation_head = self.encoder.segmentation_head
        
        # CTR regression head
        self.ctr_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(decoder_channels, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()  # CTR is between 0 and 1
        )
        
        # Cardiomegaly classification head (no sigmoid, using BCEWithLogitsLoss)
        self.cardio_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(decoder_channels, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
        
        # GradNorm parameters
        self.loss_weights = nn.Parameter(torch.ones(3))  # [seg, ctr, cardio]
        
    def forward(self, x):
        # Get encoder features with gradient checkpointing
        if self.training:
            encoder_features = checkpoint(self.encoder.encoder, x)
        else:
            encoder_features = self.encoder.encoder(x)
        
        # Get decoder features - pass encoder_features as a single argument
        decoder_output = self.encoder.decoder(encoder_features)
        # Segmentation output
        seg_output = self.segmentation_head(decoder_output)
        # CTR regression output
        ctr_output = self.ctr_head(decoder_output)
        # Cardiomegaly classification output
        cardio_output = self.cardio_head(decoder_output)
        
        return {
            'segmentation': seg_output,
            'ctr': ctr_output,
            'cardiomegaly': cardio_output
        }

    def fit(self, train_loader, val_loader, criterion, optimizer, config):
        # Initialize GradScaler for mixed precision
        scaler = GradScaler('cuda')
        
        # GradNorm optimizer for loss weights
        gradnorm_optimizer = torch.optim.Adam([self.loss_weights], lr=config.gradnorm_lr)
        
        # Track initial task losses for GradNorm
        initial_task_losses = None
        
        best_val_loss = float('inf')
        history = {
            'train_loss': [], 'val_loss': [],
            'train_iou': [], 'val_iou': [],
            'train_smape': [], 'val_smape': [],
            'train_f1': [], 'val_f1': [],
            'loss_weights': []
        }

        print("Training started...")
        patience_counter = 0
        
        for epoch in range(config.epochs):
            start_time = time.time()
            self.train()
            epoch_train_loss = 0.0
            train_metrics = initialize_metrics()
            
            # Initialize epoch-level loss accumulators for training
            epoch_train_seg_loss = 0.0
            epoch_train_ctr_loss = 0.0
            epoch_train_cardio_loss = 0.0
            
            # For GradNorm - accumulate task losses
            epoch_task_losses = torch.zeros(3, device=config.device)
            num_batches = 0

            for batch_idx, (images, masks, ctrs, cardios) in enumerate(train_loader):
                images = images.to(config.device)
                masks = masks.to(config.device).long() if masks[0] is not None else None
                ctrs = ctrs.float().to(config.device)
                cardios = cardios.float().to(config.device)

                optimizer.zero_grad()
                
                # Forward pass with mixed precision
                with autocast('cuda'):
                    outputs = self(images)

                    targets = {
                        'segmentation': masks if masks is not None else torch.zeros_like(outputs['segmentation']),
                        'ctr': ctrs.view(-1, 1),
                        'cardiomegaly': cardios.view(-1, 1)
                    }

                    loss_dict = criterion(outputs, targets, self.loss_weights)
                    loss = loss_dict['total_loss']
                
                # Backward pass with gradient scaling
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                # Accumulate losses for GradNorm
                with torch.no_grad():
                    epoch_task_losses[0] += loss_dict['seg_loss'].item()
                    epoch_task_losses[1] += loss_dict['ctr_loss'].item()
                    epoch_task_losses[2] += loss_dict['cardio_loss'].item()
                    num_batches += 1

                epoch_train_loss += loss.item()
                epoch_train_seg_loss += loss_dict['seg_loss'].item()
                epoch_train_ctr_loss += loss_dict['ctr_loss'].item()
                epoch_train_cardio_loss += loss_dict['cardio_loss'].item()
                train_metrics = update_metrics(train_metrics, outputs, targets)

            # Calculate average task losses for this epoch
            avg_task_losses = epoch_task_losses / num_batches
            
            # Initialize or update GradNorm
            if initial_task_losses is None:
                initial_task_losses = avg_task_losses.clone()
            elif epoch >= 1:  # Apply GradNorm starting from epoch 1
                # Apply GradNorm using a separate forward pass
                self.apply_gradnorm(train_loader, criterion, initial_task_losses, gradnorm_optimizer, config)

            avg_train_loss = epoch_train_loss / len(train_loader)
            avg_train_seg_loss = epoch_train_seg_loss / len(train_loader)
            avg_train_ctr_loss = epoch_train_ctr_loss / len(train_loader)
            avg_train_cardio_loss = epoch_train_cardio_loss / len(train_loader)
            train_final_metrics = compute_metrics(train_metrics)

            # Validation phase
            self.eval()
            epoch_val_loss = 0.0
            val_metrics = initialize_metrics()

            # Initialize epoch-level loss accumulators for validation
            epoch_val_seg_loss = 0.0
            epoch_val_ctr_loss = 0.0
            epoch_val_cardio_loss = 0.0

            with torch.no_grad():
                for images, masks, ctrs, cardios in val_loader:
                    images = images.to(config.device)
                    masks = masks.to(config.device) if masks[0] is not None else None
                    ctrs = ctrs.float().to(config.device)
                    cardios = cardios.float().to(config.device)

                    with autocast('cuda'):
                        outputs = self(images)
                        targets = {
                            'segmentation': masks,
                            'ctr': ctrs.view(-1, 1),
                            'cardiomegaly': cardios.view(-1, 1)
                        }

                        loss_dict = criterion(outputs, targets, self.loss_weights)
                        
                    epoch_val_loss += loss_dict['total_loss'].item()
                    epoch_val_seg_loss += loss_dict['seg_loss'].item()
                    epoch_val_ctr_loss += loss_dict['ctr_loss'].item()
                    epoch_val_cardio_loss += loss_dict['cardio_loss'].item()
                    val_metrics = update_metrics(val_metrics, outputs, targets)

            avg_val_loss = epoch_val_loss / len(val_loader)
            avg_val_seg_loss = epoch_val_seg_loss / len(val_loader)
            avg_val_ctr_loss = epoch_val_ctr_loss / len(val_loader)
            avg_val_cardio_loss = epoch_val_cardio_loss / len(val_loader)
            val_final_metrics = compute_metrics(val_metrics)

            # Store history
            history['train_loss'].append(avg_train_loss)
            history['val_loss'].append(avg_val_loss)
            history['train_iou'].append(train_final_metrics['iou'])
            history['val_iou'].append(val_final_metrics['iou'])
            history['train_smape'].append(train_final_metrics['smape'])
            history['val_smape'].append(val_final_metrics['smape'])
            history['train_f1'].append(train_final_metrics['f1'])
            history['val_f1'].append(val_final_metrics['f1'])
            history['loss_weights'].append(self.loss_weights.detach().cpu().numpy().copy())

            # Print progress with loss weights
            weights_str = f"Weights: seg={self.loss_weights[0]:.3f}, ctr={self.loss_weights[1]:.3f}, cardio={self.loss_weights[2]:.3f}"
            print(f"\nEpoch {epoch+1}/{config.epochs} | {int(time.time()-start_time)} seconds | {weights_str}")
            print(f"Train | total_loss: {avg_train_loss:.4f}, seg_loss: {avg_train_seg_loss:.4f}, "
                  f"ctr_loss: {avg_train_ctr_loss:.4f}, cardio_loss: {avg_train_cardio_loss:.4f} | "
                  f"IoU: {train_final_metrics['iou']:.4f}, sMAPE: {train_final_metrics['smape']:.2f}%, "
                  f"F1: {train_final_metrics['f1']:.4f}")
            print(f"  Val | total_loss: {avg_val_loss:.4f}, seg_loss: {avg_val_seg_loss:.4f}, "
                  f"ctr_loss: {avg_val_ctr_loss:.4f}, cardio_loss: {avg_val_cardio_loss:.4f} | "
                  f"IoU: {val_final_metrics['iou']:.4f}, sMAPE: {val_final_metrics['smape']:.2f}%, "
                  f"F1: {val_final_metrics['f1']:.4f}")

            if avg_val_loss < best_val_loss - config.min_delta:
                best_val_loss = avg_val_loss
                torch.save(self.state_dict(), config.model_save_path)
                print(f"New best validation loss: {best_val_loss:.4f}. Model saved to {config.model_save_path}")
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= config.patience:
                    print(f"Early stopping triggered at epoch {epoch+1}")
                    break
                    
        print("\nTraining complete!")
        return history

    def apply_gradnorm(self, data_loader, criterion, initial_losses, gradnorm_optimizer, config):
        """Apply GradNorm to update loss weights using a separate forward pass"""
        self.train()
        
        # Get a batch for GradNorm computation
        batch = next(iter(data_loader))
        images, masks, ctrs, cardios = batch
        
        images = images.to(config.device)
        masks = masks.to(config.device).long() if masks[0] is not None else None
        ctrs = ctrs.float().to(config.device)
        cardios = cardios.float().to(config.device)
        
        # Forward pass to get current task losses
        with autocast('cuda'):
            outputs = self(images)
            targets = {
                'segmentation': masks if masks is not None else torch.zeros_like(outputs['segmentation']),
                'ctr': ctrs.view(-1, 1),
                'cardiomegaly': cardios.view(-1, 1)
            }
            
            loss_dict = criterion(outputs, targets, torch.ones(3, device=config.device))  # Use unit weights
            
            # Get individual task losses (not weighted)
            current_losses = torch.stack([
                loss_dict['seg_loss'],
                loss_dict['ctr_loss'], 
                loss_dict['cardio_loss']
            ])
        
        # Calculate loss ratios
        loss_ratios = current_losses / initial_losses
        
        # Calculate target rates (all equal to 1.0)
        target_rates = torch.tensor(config.target_rates, device=config.device)
        
        # Calculate average loss ratio
        avg_loss_ratio = loss_ratios.mean()
        
        # Calculate relative training rates
        relative_rates = loss_ratios / avg_loss_ratio
        
        # Calculate GradNorm loss
        gradnorm_loss = torch.abs(relative_rates - target_rates).pow(config.gradnorm_alpha).sum()
        
        # Update loss weights
        gradnorm_optimizer.zero_grad()
        gradnorm_loss.backward()
        gradnorm_optimizer.step()
        
        # Normalize weights to prevent them from becoming too large
        with torch.no_grad():
            self.loss_weights.data = self.loss_weights.data / self.loss_weights.data.sum() * len(self.loss_weights)

In [6]:
class MultiTaskLoss(nn.Module):
    def __init__(self, seg_ce_weight=0.5, seg_jaccard_weight=0.5, smooth=1.0):
        super(MultiTaskLoss, self).__init__()
        self.seg_ce_weight = seg_ce_weight
        self.seg_jaccard_weight = seg_jaccard_weight
        self.smooth = smooth
        
        self.ce_loss = nn.CrossEntropyLoss()
        self.ctr_loss = nn.MSELoss()
        self.cardio_loss = nn.BCEWithLogitsLoss()  # Safe for mixed precision
    
    def forward(self, predictions, targets, loss_weights):
        device = predictions['segmentation'].device
        
        # Initialize losses
        seg_ce_loss = torch.tensor(0.0, device=device)
        seg_jaccard_loss = torch.tensor(0.0, device=device)
        
        if targets['segmentation'] is not None and targets['segmentation'].sum() > 0:
            seg_pred = predictions['segmentation']
            seg_target = targets['segmentation']
            
            # Cross Entropy Loss
            seg_ce_loss = self.ce_loss(seg_pred, seg_target)
            
            # Jaccard Loss (calculated directly)
            num_classes = seg_pred.shape[1]
            targets_onehot = F.one_hot(seg_target, num_classes=num_classes).permute(0, 3, 1, 2).float()
            probs = F.softmax(seg_pred, dim=1)
            
            intersection = (probs * targets_onehot).sum(dim=(0, 2, 3))
            total = (probs + targets_onehot).sum(dim=(0, 2, 3))
            union = total - intersection
            
            iou = (intersection + self.smooth) / (union + self.smooth)
            seg_jaccard_loss = 1 - iou.mean()
            
            # Combined segmentation loss
            seg_loss = (self.seg_ce_weight * seg_ce_loss + 
                       self.seg_jaccard_weight * seg_jaccard_loss)
        else:
            seg_loss = torch.tensor(0.0, device=device)
        
        # Other losses
        ctr_loss = self.ctr_loss(predictions['ctr'], targets['ctr'])
        cardio_loss = self.cardio_loss(predictions['cardiomegaly'], targets['cardiomegaly'])
        
        # Apply GradNorm weights
        weighted_seg_loss = loss_weights[0] * seg_loss
        weighted_ctr_loss = loss_weights[1] * ctr_loss
        weighted_cardio_loss = loss_weights[2] * cardio_loss
        
        # Total loss
        total_loss = weighted_seg_loss + weighted_ctr_loss + weighted_cardio_loss
        
        return {
            'total_loss': total_loss,
            'seg_loss': seg_loss,
            'seg_ce_loss': seg_ce_loss,
            'seg_jaccard_loss': seg_jaccard_loss,
            'ctr_loss': ctr_loss,
            'cardio_loss': cardio_loss,
            'weighted_seg_loss': weighted_seg_loss,
            'weighted_ctr_loss': weighted_ctr_loss,
            'weighted_cardio_loss': weighted_cardio_loss
        }

In [7]:
class Evaluator:
    def __init__(self, model, device):
        self.model = model
        self.device = device
    
    def evaluate(self, dataloader):
        self.model.eval()
        metrics = initialize_metrics()
        
        with torch.no_grad():
            for images, masks, ctrs, cardios in dataloader:
                images = images.to(self.device)
                masks = masks.to(self.device)
                ctrs = ctrs.float().to(self.device)
                cardios = cardios.float().to(self.device)
                
                outputs = self.model(images)
                
                # Only calculate segmentation metrics if masks contain more than background
                valid_seg = (masks > 0).any()
                targets = {
                    'segmentation': masks if valid_seg else None,
                    'ctr': ctrs.view(-1, 1),
                    'cardiomegaly': cardios.view(-1, 1)
                }
                
                metrics = update_metrics(metrics, outputs, targets)
        
        return compute_metrics(metrics)

    def evaluate_external(self, image_dir, mask_dir=None, csv_path=None, batch_size=8):
        if not os.path.exists(image_dir):
            print("External image directory not found.")
            return None

        dataset = Dataset(
            image_dir=image_dir,
            mask_dir=mask_dir if mask_dir and os.path.exists(mask_dir) else None,
            csv_path=csv_path if csv_path and os.path.exists(csv_path) else None
        )

        if len(dataset) == 0:
            print("No data found in the external dataset.")
            return None

        loader = DataLoader(dataset, batch_size=batch_size)
        metrics = self.evaluate(loader)

        print("\nExternal Dataset Results:")
        if metrics['iou'] != -1:
            print(f"IoU: {metrics['iou']:.4f}, "
                  f"sMAPE: {metrics['smape']:.2f}%, "
                  f"F1: {metrics['f1']:.4f}")
        else:
            print(f"sMAPE: {metrics['smape']:.2f}%, "
                  f"F1: {metrics['f1']:.4f}")

        return metrics

In [8]:
class Visualizer:
    def __init__(self, output_dir="plots"):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
    
    def plot_history(self, history):
        plt.figure(figsize=(15, 10))
        
        # Loss plot
        plt.subplot(2, 2, 1)
        plt.plot(history['train_loss'], label='Eğitim')
        plt.plot(history['val_loss'], label='Doğrulama')
        plt.title('Kayıp Değerleri')
        plt.xlabel('Epok')
        plt.ylabel('Kayıp')
        plt.legend()
        
        # IoU plot
        plt.subplot(2, 2, 2)
        plt.plot(history['train_iou'], label='Eğitim')
        plt.plot(history['val_iou'], label='Doğrulama')
        plt.title('Segmentasyon IoU')
        plt.xlabel('Epok')
        plt.ylabel('IoU')
        plt.legend()
        
        # sMAPE plot
        plt.subplot(2, 2, 3)
        plt.plot(history['train_smape'], label='Eğitim')
        plt.plot(history['val_smape'], label='Doğrulama')
        plt.title('KTO sMAPE (%)')
        plt.xlabel('Epok')
        plt.ylabel('sMAPE (%)')
        plt.legend()
        
        # F1 plot
        plt.subplot(2, 2, 4)
        plt.plot(history['train_f1'], label='Eğitim')
        plt.plot(history['val_f1'], label='Doğrulama')
        plt.title('Kardiyomegali F1-Skoru')
        plt.xlabel('Epok')
        plt.ylabel('F1 Skoru')
        plt.legend()
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, "training_history.png"))
        plt.show()
        plt.close()
        
    def plot_sample_predictions(self, model, dataset, device, num_samples=3, filename="predictions.png",):
        """
        Plot sample predictions with custom label colors.
        
        Args:
            model: The trained model
            dataset: The dataset to sample from
            device: Device to run inference on
            num_samples: Number of samples to plot
            filename: Output filename
        """
        label_colors = {
            0: (123, 64, 25),
            1: (255, 191, 120),
            2: (255, 238, 169)
        }
        colors = {k: (v[0]/255, v[1]/255, v[2]/255) for k, v in label_colors.items()}
        
        # Create colormap from the colors
        cmap = ListedColormap([colors[i] for i in sorted(colors.keys())])
        
        model.eval()
        # Check if any mask is non-empty to decide on number of columns
        show_gt = any(torch.any(dataset[i][1]) for i in range(len(dataset)))
        ncols = 3 if show_gt else 2
    
        fig, axes = plt.subplots(num_samples, ncols, figsize=(4 * ncols, 4 * num_samples))
        if num_samples == 1:
            axes = [axes]  # Ensure axes is always a list
    
        for i in range(num_samples):
            idx = torch.randint(0, len(dataset), (1,)).item()
            image, mask, ctr_gt, cardio_gt = dataset[idx]
            with torch.no_grad():
                pred = model(image.unsqueeze(0).to(device))
                seg_pred = torch.argmax(pred['segmentation'], dim=1).squeeze().cpu()
                ctr_pred = pred['ctr'].item()
    
            # Plot image with ground truth info in title
            axes[i][0].imshow(image[0], cmap='gray')
            axes[i][0].set_title(f"Orijinal: KTO={ctr_gt:.2f}")
            axes[i][0].axis('off')
    
            # Optional ground truth segmentation
            if show_gt:
                axes[i][1].imshow(mask, vmin=0, vmax=len(colors)-1, cmap=cmap)
                axes[i][1].set_title("Gerçek Segmentasyon")
                axes[i][1].axis('off')
    
            # Prediction
            pred_ax = axes[i][2] if show_gt else axes[i][1]
            pred_ax.imshow(seg_pred, vmin=0, vmax=len(colors)-1, cmap=cmap)
            pred_ax.set_title(f"Tahmin: KTO={ctr_pred:.2f}")
            pred_ax.axis('off')
    
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, filename))
        plt.show()
        plt.close()
        
    def plot_evaluation_results(self, metrics_dict, title="Test Sonuçları", filename="test_metrics_distribution.png"):
        labels = list(metrics_dict.keys())
        values = list(metrics_dict.values())
    
        # Skip first metric if it's negative
        if values and values[0] < 0:
            labels = labels[1:]
            values = values[1:]
    
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(labels, values, color='skyblue')
        ax.set_title(title)
        ax.set_ylabel("Skor")
    
        for i, v in enumerate(values):
            ax.text(i, v + 0.01, f"{v:.2f}", ha='center', fontweight='bold')
    
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, filename))
        plt.show()
        plt.close()

In [9]:
def initialize_metrics():
    return {
        'iou_intersections': [0, 0, 0],  # For classes 0,1,2
        'iou_unions': [0, 0, 0],
        'smape_sum': 0,
        'smape_count': 0,
        'f1_tp': 0,
        'f1_fp': 0,
        'f1_fn': 0
    }

def update_metrics(metrics, outputs, targets):
    # Segmentation IoU (only if valid masks available)
    if targets['segmentation'] is not None:
        pred_mask = torch.argmax(outputs['segmentation'], dim=1)
        true_mask = targets['segmentation']
        
        for cls in [1, 2]:  # Lung and heart classes
            pred_cls = (pred_mask == cls)
            true_cls = (true_mask == cls)
            
            intersection = (pred_cls & true_cls).sum().item()
            union = (pred_cls | true_cls).sum().item()
            
            metrics['iou_intersections'][cls] += intersection
            metrics['iou_unions'][cls] += union
    
    # CTR sMAPE
    ctr_pred = outputs['ctr'].squeeze()
    ctr_true = targets['ctr'].squeeze()
    smape = 2 * torch.abs(ctr_pred - ctr_true) / (torch.abs(ctr_pred) + torch.abs(ctr_true) + 1e-8)
    metrics['smape_sum'] += smape.sum().item()
    metrics['smape_count'] += ctr_true.numel()
    
    # Cardiomegaly F1
    cardio_pred = (outputs['cardiomegaly'].squeeze() > 0.5).float()
    cardio_true = targets['cardiomegaly'].squeeze()
    
    tp = ((cardio_pred == 1) & (cardio_true == 1)).sum().item()
    fp = ((cardio_pred == 1) & (cardio_true == 0)).sum().item()
    fn = ((cardio_pred == 0) & (cardio_true == 1)).sum().item()
    
    metrics['f1_tp'] += tp
    metrics['f1_fp'] += fp
    metrics['f1_fn'] += fn
    
    return metrics

def compute_metrics(metrics):
    """
    Calculate final metrics from accumulated values.
    Returns -1 for IoU when no segmentation masks were processed.
    """
    # Calculate IoU - handle case where no masks were available
    if metrics['iou_unions'][1] + metrics['iou_unions'][2] == 0:
        iou = -1  # Special value indicating no masks were processed
    else:
        ious = []
        for cls in [1, 2]:  # Only lung (1) and heart (2) classes
            if metrics['iou_unions'][cls] > 0:
                iou_val = metrics['iou_intersections'][cls] / metrics['iou_unions'][cls]
                ious.append(iou_val)
        # Calculate mean IoU only if we have valid classes
        iou = sum(ious) / len(ious) if ious else 0.0

    # Calculate sMAPE (always available if CTR values exist)
    smape = (metrics['smape_sum'] / metrics['smape_count']) * 100 if metrics['smape_count'] > 0 else 0.0
    
    # Calculate F1 score (always available if cardiomegaly labels exist)
    precision = metrics['f1_tp'] / (metrics['f1_tp'] + metrics['f1_fp'] + 1e-8)
    recall = metrics['f1_tp'] / (metrics['f1_tp'] + metrics['f1_fn'] + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    
    return {
        'iou': iou,
        'smape': smape,
        'f1': f1
    }

def calculate_iou(pred, target, num_classes=3):
    """Compute IoU for segmentation masks (ignore background)"""
    ious = []
    pred = torch.argmax(pred, dim=1)  # (B, H, W)
    for cls in range(1, num_classes):  # Skip background
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        intersection = (pred_cls & target_cls).sum()
        union = (pred_cls | target_cls).sum()
        ious.append((intersection + 1e-6) / (union + 1e-6))
    return torch.mean(torch.tensor(ious))

def calculate_smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error for CTR"""
    return 2 * torch.mean(torch.abs(y_pred - y_true) / (torch.abs(y_pred) + torch.abs(y_true) + 1e-8)) * 100

def calculate_f1(y_true, y_pred_prob, threshold=0.5):
    """F1-score for cardiomegaly classification"""
    y_pred = (y_pred_prob > threshold).float()
    tp = (y_pred * y_true).sum()
    fp = ((1 - y_true) * y_pred).sum()
    fn = (y_true * (1 - y_pred)).sum()
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    return 2 * precision * recall / (precision + recall + 1e-8)

# pipeline

In [10]:
config = Config()

# Internal dataset
full_dataset = Dataset(
    image_dir=os.path.join(config.data_dir, "images"),
    mask_dir=os.path.join(config.data_dir, "masks")
)

# Split
train_size = int(len(full_dataset) * config.train_ratio)
val_size = int(len(full_dataset) * config.val_ratio)
test_size = len(full_dataset) - train_size - val_size

train_set, val_set, test_set = random_split(full_dataset, [train_size, val_size, test_size])

# DataLoaders
train_loader = DataLoader(train_set, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=config.batch_size)
test_loader = DataLoader(test_set, batch_size=config.batch_size)

In [ ]:
model = MultiTaskUNet().to(config.device)

criterion = MultiTaskLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

history = model.fit(train_loader, val_loader, criterion, optimizer, config)

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.5M [00:00<?, ?B/s]

Training started...


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(



Epoch 1/100 | 308 seconds | Weights: seg=0.983, ctr=0.975, cardio=0.972
Train | total_loss: 1.1494, seg_loss: 0.4815, ctr_loss: 0.0062, cardio_loss: 0.6745 | IoU: 0.5507, sMAPE: 13.52%, F1: 0.0000
  Val | total_loss: 0.8833, seg_loss: 0.2557, ctr_loss: 0.0055, cardio_loss: 0.6445 | IoU: 0.8109, sMAPE: 12.75%, F1: 0.0000
New best validation loss: 0.8833. Model saved to efficient-b1_unet_on_all_synthetic_cleaned_traditional_brightness_data.pth

Epoch 2/100 | 261 seconds | Weights: seg=1.017, ctr=0.996, cardio=0.987
Train | total_loss: 0.7280, seg_loss: 0.1473, ctr_loss: 0.0046, cardio_loss: 0.6042 | IoU: 0.8910, sMAPE: 11.34%, F1: 0.0000
  Val | total_loss: 0.7177, seg_loss: 0.1397, ctr_loss: 0.0054, cardio_loss: 0.5777 | IoU: 0.8768, sMAPE: 13.02%, F1: 0.0000
New best validation loss: 0.7177. Model saved to efficient-b1_unet_on_all_synthetic_cleaned_traditional_brightness_data.pth

Epoch 3/100 | 260 seconds | Weights: seg=1.023, ctr=0.996, cardio=0.981
Train | total_loss: 0.5592, seg_l

In [ ]:
visualizer = Visualizer(output_dir="/kaggle/working/plots_efficient-b1_unet_on_all_synthetic_cleaned_traditional_brightness_data")

In [ ]:
visualizer.plot_history(history)

In [ ]:
model.load_state_dict(torch.load(config.model_save_path))
model.eval()

evaluator = Evaluator(model, config.device)
test_metrics = evaluator.evaluate(test_loader)

print("\nTest Set Results:")
print(f"IoU: {test_metrics['iou']:.4f}, "
      f"sMAPE: {test_metrics['smape']:.2f}%, "
      f"F1: {test_metrics['f1']:.4f}")

visualizer.plot_evaluation_results(test_metrics, title="Dahili Test Sonuçları", filename="test_metrics_distribution_internal.png")
visualizer.plot_sample_predictions(model, test_set, config.device, num_samples=20, filename="predictions_internal.png")

In [ ]:
if config.external_data_dir:
    image_dir = os.path.join(config.external_data_dir, "images")
    mask_dir = os.path.join(config.external_data_dir, "masks")
    csv_path = os.path.join(config.external_data_dir, "data.csv")
    
    external_metrics = evaluator.evaluate_external(
        image_dir=image_dir,
        mask_dir=mask_dir,
        csv_path=csv_path,
        batch_size=config.batch_size
    )

    if external_metrics:
        visualizer.plot_evaluation_results(external_metrics, title="Harici Test Sonuçları", filename="test_metrics_distribution_external.png")

        # External prediction samples
        external_dataset = Dataset(image_dir=image_dir, mask_dir=mask_dir, csv_path=csv_path)
        visualizer.plot_sample_predictions(model, external_dataset, config.device, num_samples=20, filename="predictions_external.png")